# PRISM-V2: Class-Conditional, Interaction-Aware Progressive Pruning

**Runnable research prototype; accuracy gains are hypotheses, not guarantees.** Uses CIFAR-10 ResNet-56 (option-A residual shortcuts), actual structural channel removal, a teacher, class-conditional gradient damage measurements, limited candidate pair-interaction tests, validation-based rollback, and distillation. The test set is reserved for final metrics.

**Research caveats:** Class-conditional importance and pair interactions have precedents; novelty must be established through exact related-work comparisons and an ablation study. Per-candidate recovery probes are expensive, so this notebook measures final round recoverability rather than training every candidate group separately. FLOPs uses **2 operations per multiply-accumulate**, whereas many CIFAR papers report MACs (half this value). Do not compare FLOPs across counting conventions.

**Start:** set `C.smoke_test=True` in configuration for a minimal end-to-end check, then reset and rerun with `False`. Training from scratch is computationally intensive.


In [ ]:
# Optional environment setup (uncomment if necessary)
# %pip install -q torch torchvision numpy matplotlib pandas scikit-learn tqdm thop pynvml
# Restart kernel after installation if required.

In [ ]:
import os, json, math, time, random, logging, traceback, copy, hashlib, platform, subprocess, warnings
from pathlib import Path
from dataclasses import dataclass, asdict
from contextlib import nullcontext
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from tqdm.auto import tqdm

@dataclass
class Config:
    seed: int = 42
    data_dir: str = './data'
    run_dir: str = './prism_runs/prism_v2_resnet56_c10'
    smoke_test: bool = False # set True for a fast functional check
    batch_size: int = 128
    num_workers: int = 2 if os.name != 'nt' else 0
    train_epochs: int = 200
    recovery_epochs: int = 40
    prune_rounds: int = 3
    # successive fractions of ORIGINAL channels retained; no shape assumptions
    stage_target_ratios: tuple = (0.85, 0.70, 0.58)
    mid_target_ratios: tuple = (0.80, 0.65, 0.52)
    min_stage_channels: int = 8
    min_mid_channels: int = 8
    calibration_batches: int = 8
    val_fraction: float = 0.10
    val_check_batches: int = 0 # 0 = all validation batches
    rollback_tolerance_pp: float = 0.50
    lr: float = 0.1
    recovery_lr: float = 0.025
    weight_decay: float = 1e-4
    momentum: float = 0.9
    kd_temperature: float = 5.0
    kd_weight: float = 0.7
    relation_weight: float = 0.05
    score_gradient_weight: float = 0.60
    score_redundancy_weight: float = 0.20
    class_sensitivity_weight: float = 0.35
    interaction_weight: float = 0.30
    interaction_candidates: int = 3 # only the lowest scoring channels in a block
    interaction_batches: int = 1 # small held-out TRAIN subset; not test images
    interaction_modules_per_round: int = 8 # cap interaction search cost
    min_accuracy_before_accept: float = 0.0 # validation criterion in percent
    latency_repetitions: int = 100
    gpu_power_watts_estimate: float = 0.0 # 0 means no fabricated energy estimate

C = Config()
# C.smoke_test = True  # uncomment for verification before the full run
if C.smoke_test:
    C.train_epochs=1; C.recovery_epochs=1; C.prune_rounds=1
    C.calibration_batches=1; C.latency_repetitions=10; C.val_check_batches=3
    C.interaction_modules_per_round=2; C.interaction_candidates=2
    C.batch_size=64
RUN = Path(C.run_dir).resolve(); RUN.mkdir(parents=True,exist_ok=True)
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
random.seed(C.seed); np.random.seed(C.seed); torch.manual_seed(C.seed)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(C.seed)
torch.backends.cudnn.benchmark=True
logger=logging.getLogger('prism'); logger.setLevel(logging.INFO); logger.handlers.clear()
fh=logging.FileHandler(RUN/'run.log',encoding='utf-8'); sh=logging.StreamHandler()
for h in (fh,sh): h.setFormatter(logging.Formatter('%(asctime)s | %(levelname)s | %(message)s'));logger.addHandler(h)
def log_exception(where,e):
    with open(RUN/'errors.log','a',encoding='utf-8') as f:
        f.write(f'\n[{time.strftime("%Y-%m-%d %H:%M:%S")}] {where}: {e}\n{traceback.format_exc()}\n')
    logger.exception('%s failed',where)
def atomic_save(obj,path):
    path=Path(path);temp=path.with_suffix(path.suffix+'.tmp');torch.save(obj,temp);os.replace(temp,path)
def append_jsonl(path,obj):
    with open(path,'a',encoding='utf-8') as f:f.write(json.dumps(obj,default=float)+'\n')
logger.info('Device=%s; torch=%s; smoke_test=%s',DEVICE,torch.__version__,C.smoke_test)
with open(RUN/'config.json','w') as f:json.dump(asdict(C),f,indent=2)

## Data: deterministic train/validation split, test held out

Training augmentation: random crop and flip. Validation and test: normalization only. A local CIFAR-10 cache is reused automatically; downloads occur only if unavailable.

In [ ]:
mean=(0.4914,0.4822,0.4465); std=(0.2470,0.2435,0.2616)
train_tf=transforms.Compose([transforms.RandomCrop(32,padding=4),transforms.RandomHorizontalFlip(),transforms.ToTensor(),transforms.Normalize(mean,std)])
eval_tf=transforms.Compose([transforms.ToTensor(),transforms.Normalize(mean,std)])
try:
    full_aug=datasets.CIFAR10(C.data_dir,train=True,download=True,transform=train_tf)
    full_clean=datasets.CIFAR10(C.data_dir,train=True,download=False,transform=eval_tf)
    test_data=datasets.CIFAR10(C.data_dir,train=False,download=True,transform=eval_tf)
except Exception as e:
    log_exception('Dataset initialization',e);raise
rng=np.random.default_rng(C.seed)
indices=rng.permutation(len(full_aug)); nval=int(len(indices)*C.val_fraction)
tr_idx=indices[nval:].tolist(); va_idx=indices[:nval].tolist()
if C.smoke_test:
    tr_idx=tr_idx[:2048]; va_idx=va_idx[:512]; test_data=Subset(test_data, list(range(512)))
train_ds=Subset(full_aug,tr_idx);val_ds=Subset(full_clean,va_idx)
def dataloader(ds,shuffle=False,bs=None):
    return DataLoader(ds,batch_size=bs or C.batch_size,shuffle=shuffle,num_workers=C.num_workers,pin_memory=DEVICE.type=='cuda',persistent_workers=C.num_workers>0)
train_loader=dataloader(train_ds,True);val_loader=dataloader(val_ds);test_loader=dataloader(test_data)
cal_loader=dataloader(Subset(full_clean,tr_idx[:min(2048,len(tr_idx))]),False)
logger.info('Train=%d validation=%d test=%d',len(train_ds),len(val_ds),len(test_data))

## CIFAR ResNet-56 with removable stage-wide and internal channels

The original uses 16/32/64 stage widths, 9 basic blocks per stage, and **option-A shortcuts**. For a compressed network, channel IDs are tracked in the coordinates of the original model: a transition shortcut copies only surviving channels from the original centrally padded shortcut, leaving the remaining outputs zero. This guarantees correct tensor alignment and allows direct weight inheritance. Conv layers have no biases and are followed by BN.

In [ ]:
class IndexedOptionA(nn.Module):
    def __init__(self,source_ids,dest_ids,source_total,dest_total,stride=1):
        super().__init__();self.stride=stride
        offset=(dest_total-source_total)//2
        locations={old:i for i,old in enumerate(source_ids)}
        source=[];dest=[]
        for k,d in enumerate(dest_ids):
            old_source=int(d)-offset
            if old_source in locations:
                source.append(locations[old_source]);dest.append(k)
        self.register_buffer('src',torch.tensor(source,dtype=torch.long),persistent=False)
        self.register_buffer('dst',torch.tensor(dest,dtype=torch.long),persistent=False)
        self.channels=len(dest_ids)
    def forward(self,x):
        if self.stride==2:x=F.avg_pool2d(x,2,2)
        out=x.new_zeros((x.shape[0],self.channels,x.shape[2],x.shape[3]))
        if self.src.numel():out[:,self.dst,:,:]=x[:,self.src,:,:]
        return out

class BasicBlock(nn.Module):
    def __init__(self,in_ids,out_ids,mid_ids,in_total,out_total,stride):
        super().__init__(); self.conv1=nn.Conv2d(len(in_ids),len(mid_ids),3,stride,1,bias=False)
        self.bn1=nn.BatchNorm2d(len(mid_ids))
        self.conv2=nn.Conv2d(len(mid_ids),len(out_ids),3,1,1,bias=False)
        self.bn2=nn.BatchNorm2d(len(out_ids))
        self.shortcut=IndexedOptionA(in_ids,out_ids,in_total,out_total,stride)
    def forward(self,x):
        z=F.relu(self.bn1(self.conv1(x)),inplace=False)
        z=self.bn2(self.conv2(z));return F.relu(z+self.shortcut(x),inplace=False)

class CIFARResNet56(nn.Module):
    def __init__(self,selection=None):
        super().__init__()
        if selection is None:
            selection={'stages':[list(range(n)) for n in (16,32,64)],
                       'mid':[[list(range(n)) for _ in range(9)] for n in (16,32,64)]}
        self.selection=copy.deepcopy(selection)
        stages=selection['stages'];mid=selection['mid']
        self.stem=nn.Conv2d(3,len(stages[0]),3,1,1,bias=False)
        self.stem_bn=nn.BatchNorm2d(len(stages[0]));self.groups=nn.ModuleList()
        for s in range(3):
            blocks=nn.ModuleList()
            for j in range(9):
                prev=stages[s-1] if (s>0 and j==0) else stages[s]
                prev_total=(16,32,64)[s-1] if (s>0 and j==0) else (16,32,64)[s]
                stride=2 if (s>0 and j==0) else 1
                blocks.append(BasicBlock(prev,stages[s],mid[s][j],prev_total,(16,32,64)[s],stride))
            self.groups.append(blocks)
        self.fc=nn.Linear(len(stages[-1]),10)
    def forward_features(self,x):
        x=F.relu(self.stem_bn(self.stem(x)),inplace=False)
        for group in self.groups:
            for block in group:x=block(x)
        return F.adaptive_avg_pool2d(x,1).flatten(1)
    def forward(self,x):return self.fc(self.forward_features(x))

def copy_conv(src,dst,out_ids,in_ids):
    with torch.no_grad():
        dst.weight.copy_(src.weight.data.index_select(0,torch.tensor(out_ids,device=src.weight.device)).index_select(1,torch.tensor(in_ids,device=src.weight.device)))
def copy_bn(src,dst,idx):
    with torch.no_grad():
        index=torch.tensor(idx,device=src.weight.device)
        for nm in ('weight','bias','running_mean','running_var'):
            getattr(dst,nm).copy_(getattr(src,nm).data.index_select(0,index))
        dst.num_batches_tracked.copy_(src.num_batches_tracked)

def inherit_weights(original,new):
    # IDs for every retained channel refer to their corresponding original architecture IDs.
    src_sel=original.selection;dst_sel=new.selection
    # Original may itself be compressed; translate indices to positions in the source model.
    def pos(ids,orig_ids):
        loc={v:k for k,v in enumerate(orig_ids)}
        if any(i not in loc for i in ids):raise ValueError('Candidate cannot add a channel absent from the source')
        return [loc[i] for i in ids]
    si=pos(dst_sel['stages'][0],src_sel['stages'][0])
    copy_conv(original.stem,new.stem,si,[0,1,2]);copy_bn(original.stem_bn,new.stem_bn,si)
    for s in range(3):
        oi=pos(dst_sel['stages'][s],src_sel['stages'][s])
        for j in range(9):
            src=original.groups[s][j];dst=new.groups[s][j]
            pi=pos(dst_sel['stages'][s-1],src_sel['stages'][s-1]) if s and j==0 else oi
            mi=pos(dst_sel['mid'][s][j],src_sel['mid'][s][j])
            copy_conv(src.conv1,dst.conv1,mi,pi);copy_bn(src.bn1,dst.bn1,mi)
            copy_conv(src.conv2,dst.conv2,oi,mi);copy_bn(src.bn2,dst.bn2,oi)
    with torch.no_grad():
        new.fc.weight.copy_(original.fc.weight[:,pos(dst_sel['stages'][-1],src_sel['stages'][-1])]);new.fc.bias.copy_(original.fc.bias)
    return new

full_model=CIFARResNet56().to(DEVICE)
assert full_model(torch.randn(2,3,32,32,device=DEVICE)).shape==(2,10)
logger.info('Original model parameters: %d',sum(p.numel() for p in full_model.parameters()))

## Training, checkpointing and evaluation

The best validation checkpoint is saved after each epoch, and the latest training checkpoint also includes optimizer and scheduler state for crash recovery. A terminal checkpoint records each pruning round. Test data remains untouched until final evaluation.

In [ ]:
def model_parameters(model):return sum(x.numel() for x in model.parameters())
def accuracy_and_loss(model,loader,max_batches=0,collect=False):
    model.eval();total=correct=0;loss_sum=0.;ys=[];preds=[];probs=[]
    with torch.inference_mode():
        for i,(x,y) in enumerate(loader):
            if max_batches and i>=max_batches:break
            x=x.to(DEVICE,non_blocking=True);y=y.to(DEVICE,non_blocking=True)
            z=model(x);loss_sum+=F.cross_entropy(z,y,reduction='sum').item()
            pred=z.argmax(1);correct+=(pred==y).sum().item();total+=y.numel()
            if collect:ys.extend(y.cpu().tolist());preds.extend(pred.cpu().tolist());probs.extend(z.softmax(1).cpu().tolist())
    out={'acc':100*correct/max(1,total),'loss':loss_sum/max(1,total),'n':total}
    if collect:out.update(y_true=ys,y_pred=preds,probabilities=probs)
    return out

def relational_loss(student,teacher):
    # Pairwise sample similarity in batch; works even if student feature width differs.
    a=F.normalize(student.float(),dim=1);b=F.normalize(teacher.float(),dim=1)
    return F.mse_loss(a@a.T,b@b.T)

def train_epoch(model,loader,opt,teacher=None,epoch=0):
    model.train();teacher.eval() if teacher is not None else None
    cumulative=0.;total=0;correct=0
    for x,y in loader:
        x=x.to(DEVICE,non_blocking=True);y=y.to(DEVICE,non_blocking=True)
        opt.zero_grad(set_to_none=True);z=model(x);ce=F.cross_entropy(z,y)
        loss=ce
        if teacher is not None:
            with torch.no_grad():tz=teacher(x);tf=teacher.forward_features(x)
            sf=model.forward_features(x);temp=C.kd_temperature
            kd=F.kl_div(F.log_softmax(z/temp,dim=1),F.softmax(tz/temp,dim=1),reduction='batchmean')*(temp**2)
            loss=(1-C.kd_weight)*ce+C.kd_weight*kd+C.relation_weight*relational_loss(sf,tf)
        if not torch.isfinite(loss):raise FloatingPointError(f'Nonfinite training loss at epoch {epoch}')
        loss.backward();torch.nn.utils.clip_grad_norm_(model.parameters(),max_norm=5.0);opt.step()
        cumulative+=loss.item()*y.numel();total+=y.numel();correct+=(z.argmax(1)==y).sum().item()
    return {'train_loss':cumulative/max(1,total),'train_acc':100*correct/max(1,total)}

def fit(model,tag,epochs,lr,teacher=None,resume=True):
    latest=RUN/f'{tag}_latest.pt';bestfile=RUN/f'{tag}_best.pt'
    optimizer=torch.optim.SGD(model.parameters(),lr=lr,momentum=C.momentum,weight_decay=C.weight_decay,nesterov=True)
    scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=max(1,epochs))
    start=0;best=-1.;history=[]
    if resume and latest.exists():
        ck=torch.load(latest,map_location=DEVICE,weights_only=False)
        if ck.get('selection')==model.selection and ck.get('total_epochs')==epochs:
            model.load_state_dict(ck['model']);optimizer.load_state_dict(ck['optimizer']);scheduler.load_state_dict(ck['scheduler'])
            start=ck['epoch']+1;best=ck['best_acc'];history=ck['history'];logger.info('Resumed %s from epoch %d',tag,start)
        else:logger.warning('Ignored incompatible training checkpoint %s',latest)
    for ep in range(start,epochs):
        try:
            tr=train_epoch(model,train_loader,optimizer,teacher,ep)
            va=accuracy_and_loss(model,val_loader,C.val_check_batches)
            record={'tag':tag,'epoch':ep+1,**tr,'val_acc':va['acc'],'val_loss':va['loss'],'lr':optimizer.param_groups[0]['lr']}
            history.append(record);append_jsonl(RUN/'training_history.jsonl',record)
            if va['acc']>best:
                best=va['acc'];atomic_save({'model':model.state_dict(),'selection':model.selection,'val_acc':best,'epoch':ep},bestfile)
            scheduler.step()
            atomic_save({'model':model.state_dict(),'optimizer':optimizer.state_dict(),'scheduler':scheduler.state_dict(),
                         'selection':model.selection,'epoch':ep,'total_epochs':epochs,'best_acc':best,'history':history},latest)
            logger.info('%s epoch %d/%d train %.2f%% val %.2f%% best %.2f%%',tag,ep+1,epochs,tr['train_acc'],va['acc'],best)
        except Exception as e:log_exception(f'fit {tag} epoch {ep}',e);raise
    if bestfile.exists():model.load_state_dict(torch.load(bestfile,map_location=DEVICE,weights_only=False)['model'])
    return model,history,best

## FLOPs, latency, energy and model-size utilities

FLOPs are counted with a self-contained forward-hook counter (two FLOPs per convolutional multiply–accumulate). Latency measurements include warm-up, CUDA synchronization and device-specific timing. Energy is measured by optional NVML power sampling if supported; otherwise it is reported as unavailable, not invented.

In [ ]:
def count_flops(model):
    totals=[];hooks=[]
    def hook(module,args,out):
        x=args[0];b,co,h,w=out.shape;cin=module.in_channels;k=module.kernel_size
        totals.append(2*b*co*h*w*(cin//module.groups)*k[0]*k[1])
    def fc_hook(module,args,out):totals.append(2*args[0].shape[0]*module.in_features*module.out_features)
    for m in model.modules():
        if isinstance(m,nn.Conv2d):hooks.append(m.register_forward_hook(hook))
        if isinstance(m,nn.Linear):hooks.append(m.register_forward_hook(fc_hook))
    was_training=model.training;model.eval()
    try:
        with torch.inference_mode():model(torch.zeros(1,3,32,32,device=DEVICE))
    finally:
        for h in hooks:h.remove()
        model.train(was_training)
    return int(sum(totals))

def serialized_size_mb(model):
    import io
    f=io.BytesIO();torch.save(model.state_dict(),f);return f.tell()/1048576

def measure_latency(model,reps=None):
    model.eval();reps=reps or C.latency_repetitions
    x=torch.randn(1,3,32,32,device=DEVICE)
    with torch.inference_mode():
        for _ in range(20 if not C.smoke_test else 3):model(x)
        if DEVICE.type=='cuda':torch.cuda.synchronize()
        durations=[]
        for _ in range(reps):
            if DEVICE.type=='cuda':torch.cuda.synchronize()
            st=time.perf_counter();model(x)
            if DEVICE.type=='cuda':torch.cuda.synchronize()
            durations.append((time.perf_counter()-st)*1000)
    return {'latency_mean_ms':float(np.mean(durations)),'latency_median_ms':float(np.median(durations)),
            'latency_p95_ms':float(np.percentile(durations,95))}

def measure_resources(model,do_latency=False):
    d={'parameters':model_parameters(model),'flops':count_flops(model),'size_mb':serialized_size_mb(model)}
    if do_latency:d.update(measure_latency(model))
    d['gpu_peak_mb']=float(torch.cuda.max_memory_allocated()/1048576) if DEVICE.type=='cuda' else None
    d['energy_mj_per_inference']=None # not reported unless power instrumentation is actually implemented
    return d

## Train the teacher, or load its completed checkpoint

In [ ]:
try:
    teacher,base_history,base_best=fit(full_model,'baseline',C.train_epochs,C.lr,teacher=None,resume=True)
    teacher.eval();baseline_val=accuracy_and_loss(teacher,val_loader,C.val_check_batches)
    baseline_resources=measure_resources(teacher)
    logger.info('Teacher validation accuracy=%.3f%%; resources=%s',baseline_val['acc'],baseline_resources)
except Exception as e:log_exception('Teacher training',e);raise

## PRISM-V2: class-conditioned gradients and group interaction probes

**Class damage estimate:** for each example, measure `abs(activation × gradient)` per channel, pooled spatially; accumulate by true CIFAR-10 class. Channel importance combines the mean and worst-class sensitivity (worst class reduces the risk of deleting features used for a minority of difficult examples). This is a **first-order proxy**, not a true masked-loss measurement.

**Pair interaction:** within a bounded set of candidate modules, evaluate masking of channel *i*, channel *j*, and both simultaneously on a fixed, class-mixed training-calibration minibatch. Positive superadditive loss means the pair should not be simultaneously pruned. We protect one member by raising its keep score. We never evaluate on the test set during pruning.


In [ ]:
def normalize01(x):
    x=np.asarray(x,dtype=np.float64)
    return (x-x.min())/(x.max()-x.min()+1e-9)

def score_channels(model,loader,max_batches):
    model.eval();records={};handlers=[];labels_state={'y':None}
    module_map={}
    for s in range(3):
        module_map[f'stage{s}']=model.groups[s][-1]
        for j in range(9):module_map[f'mid{s}_{j}']=model.groups[s][j].bn1
    def mk_hook(key):
        def hook(mod,args,out):
            if not out.requires_grad:return
            act=out.detach(); pooled=F.adaptive_avg_pool2d(act,1).flatten(1)
            state=records.setdefault(key,{'prop':None,'grad':None,'vectors':[],
                'count':0,'class_sum':None,'class_count':np.zeros(10,dtype=np.int64)})
            prop=act.square().mean(dim=(0,2,3)).cpu().double()
            state['prop']=prop if state['prop'] is None else state['prop']+prop
            state['vectors'].append(pooled.cpu().float());state['count']+=1
            def backward_hook(grad):
                # Gradients retain a batch dimension. Compute per-image Taylor proxy.
                per_image=(act*grad.detach()).sum(dim=(2,3)).abs().cpu().double()
                imp=per_image.mean(0)
                state['grad']=imp if state['grad'] is None else state['grad']+imp
                if state['class_sum'] is None:
                    state['class_sum']=torch.zeros(10,per_image.shape[1],dtype=torch.float64)
                labels=labels_state['y']
                for k in range(10):
                    mask=labels==k
                    if mask.any():
                        state['class_sum'][k]+=per_image[mask].sum(dim=0)
                        state['class_count'][k]+=int(mask.sum())
            out.register_hook(backward_hook)
        return hook
    for key,module in module_map.items():handlers.append(module.register_forward_hook(mk_hook(key)))
    try:
        for i,(x,y) in enumerate(loader):
            if i>=max_batches:break
            x=x.to(DEVICE);y=y.to(DEVICE);labels_state['y']=y.detach().cpu()
            model.zero_grad(set_to_none=True)
            F.cross_entropy(model(x),y).backward()
    finally:
        for h in handlers:h.remove()
        model.zero_grad(set_to_none=True)
    scores={};diagnostics={}
    for key,r in records.items():
        g=(r['grad']/r['count']).numpy();p=(r['prop']/r['count']).numpy()
        v=torch.cat(r['vectors'],0).numpy();v=v-v.mean(axis=0,keepdims=True)
        v=v/(np.linalg.norm(v,axis=0,keepdims=True)+1e-7)
        sim=np.abs(v.T@v);np.fill_diagonal(sim,0);redund=sim.max(axis=1)
        cls=(r['class_sum'].numpy() / np.maximum(1,r['class_count'])[:,None])
        present=r['class_count']>0
        cls=cls[present] if present.any() else cls
        # Normalized classwise estimates prevent common classes dominating.
        class_imp=0.5*np.mean(cls,axis=0)+0.5*np.max(cls,axis=0)
        score=(C.score_gradient_weight*normalize01(g)+
               (1-C.score_gradient_weight)*normalize01(p)+
               C.class_sensitivity_weight*normalize01(class_imp)-
               C.score_redundancy_weight*normalize01(redund))
        ids=(model.selection['stages'][int(key[-1])] if key.startswith('stage')
             else model.selection['mid'][int(key[3])][int(key.split('_')[1])])
        scores[key]=dict(zip(ids,score.tolist()))
        diagnostics[key]={'classes_observed':r['class_count'].tolist(),
                          'max_pair_correlation':float(np.max(redund)),
                          'importance_mean':float(np.mean(class_imp))}
    return scores,diagnostics

def _module_for_key(model,key):
    if key.startswith('stage'):return model.groups[int(key[-1])][-1]
    s,j=map(int,key[3:].split('_'))
    return model.groups[s][j].bn1

def _masked_loss(model,x,y,module,masked_ids):
    # Hook output tensor rather than overwriting parameters. Used ONLY for
    # temporary candidate evaluation; accepted networks are physically rebuilt.
    ids=torch.as_tensor(masked_ids,device=x.device,dtype=torch.long)
    def apply(mod,inp,out):
        gate=torch.ones(out.shape[1],device=out.device,dtype=out.dtype)
        gate[ids]=0
        return out*gate.view(1,-1,1,1)
    h=module.register_forward_hook(apply)
    try:
        with torch.inference_mode():return float(F.cross_entropy(model(x),y).item())
    finally:h.remove()

def interaction_protection(model,scores,loader):
    model.eval();x,y=next(iter(loader));x=x.to(DEVICE);y=y.to(DEVICE)
    with torch.inference_mode():base=float(F.cross_entropy(model(x),y).item())
    # Restrict probes to modules with low-scoring candidates, to control runtime.
    ranked=sorted(scores,key=lambda k:min(scores[k].values()))
    chosen=ranked[:C.interaction_modules_per_round]
    logs=[]
    for key in chosen:
        ids=model.selection['stages'][int(key[-1])] if key.startswith('stage') else model.selection['mid'][int(key[3])][int(key.split('_')[1])]
        selected=sorted(ids,key=lambda z:scores[key][z])[:C.interaction_candidates]
        if len(selected)<2:continue
        module=_module_for_key(model,key)
        positions={channel:i for i,channel in enumerate(ids)}
        singles={i:_masked_loss(model,x,y,module,[positions[i]])-base for i in selected}
        for idx,i in enumerate(selected):
            for j in selected[idx+1:]:
                both=_masked_loss(model,x,y,module,[positions[i],positions[j]])-base
                interaction=both-singles[i]-singles[j]
                if interaction>0:
                    # Protect the channel with stronger initial keep score.
                    protect=i if scores[key][i]>=scores[key][j] else j
                    scores[key][protect]+=C.interaction_weight*interaction/(abs(base)+1e-6)
                logs.append({'module':key,'channel_i':i,'channel_j':j,
                    'loss_delta_i':singles[i],'loss_delta_j':singles[j],
                    'interaction':interaction})
    return scores,logs

def target_selection(old,scores,round_number):
    rstage=C.stage_target_ratios[min(round_number,len(C.stage_target_ratios)-1)]
    rmid=C.mid_target_ratios[min(round_number,len(C.mid_target_ratios)-1)]
    new=copy.deepcopy(old)
    def select(ids,score,base,min_count,ratio):
        n=max(min_count,int(round(base*ratio)));n=min(n,len(ids))
        selected=set(sorted(ids,key=lambda z:(-score.get(z,0),z))[:n])
        return sorted(selected)
    for s,total in enumerate((16,32,64)):
        new['stages'][s]=select(old['stages'][s],scores[f'stage{s}'],total,C.min_stage_channels,rstage)
        for j in range(9):
            new['mid'][s][j]=select(old['mid'][s][j],scores[f'mid{s}_{j}'],total,C.min_mid_channels,rmid)
    return new

## Progressive compression, recovery, acceptance and rollback

At each round: compute scores → select a physically smaller architecture → inherit matching weights → train with teacher KD → accept if its validation accuracy is within `rollback_tolerance_pp` of the currently accepted model. If rejected, the old model is retained. Progress is saved after each round so interrupted runs can resume. The acceptance tolerance controls how much accuracy can be sacrificed; it is an experimental hyperparameter, not a proof of optimality.

In [ ]:
progress_file=RUN/'pruning_progress.pt';round_history=[]
student=copy.deepcopy(teacher).to(DEVICE);accepted_val=baseline_val['acc'];start_round=0
if progress_file.exists():
    try:
        progress=torch.load(progress_file,map_location=DEVICE,weights_only=False)
        if progress.get('planned_rounds')==C.prune_rounds and progress.get('teacher_params')==model_parameters(teacher):
            student=CIFARResNet56(progress['selection']).to(DEVICE)
            student.load_state_dict(progress['model']);accepted_val=progress['accepted_val']
            round_history=progress['history'];start_round=progress['next_round']
            logger.info('Resuming compression from round %d',start_round+1)
        else:logger.warning('Ignoring incompatible compression progress')
    except Exception as e:log_exception('Loading pruning progress',e);raise

for ri in range(start_round,C.prune_rounds):
    try:
        logger.info('Beginning prune round %d/%d',ri+1,C.prune_rounds)
        scores,class_diagnostics=score_channels(student,cal_loader,C.calibration_batches)
        scores,pair_logs=interaction_protection(student,scores,cal_loader)
        append_jsonl(RUN/'class_diagnostics.jsonl',{'round':ri+1,'details':class_diagnostics})
        for row in pair_logs:append_jsonl(RUN/'pair_interactions.jsonl',{'round':ri+1,**row})
        append_jsonl(RUN/'score_stats.jsonl',{'round':ri+1,'score_ranges':{k:[min(v.values()),max(v.values())] for k,v in scores.items()}})
        selection=target_selection(student.selection,scores,ri)
        if selection==student.selection:
            logger.warning('Round %d has no further removable channels',ri+1);break
        proposed=inherit_weights(student,CIFARResNet56(selection).to(DEVICE))
        proposed_before=accuracy_and_loss(proposed,val_loader,C.val_check_batches)
        proposed,rec_hist,recovered_acc=fit(proposed,f'round{ri+1}',C.recovery_epochs,C.recovery_lr,teacher=teacher,resume=True)
        after=accuracy_and_loss(proposed,val_loader,C.val_check_batches)
        accepted=(after['acc']>=max(C.min_accuracy_before_accept,accepted_val-C.rollback_tolerance_pp))
        rec={'round':ri+1,'accepted':accepted,'before_val_acc':proposed_before['acc'],
             'after_val_acc':after['acc'],'previous_val_acc':accepted_val,
             **measure_resources(proposed)}
        round_history.append(rec);append_jsonl(RUN/'round_history.jsonl',rec)
        if accepted:
            student=proposed;accepted_val=after['acc']
            atomic_save({'model':student.state_dict(),'selection':student.selection,'val_acc':accepted_val},RUN/'best_compressed.pt')
        else:logger.warning('Rejected round %d; reverted to previously accepted network',ri+1)
        atomic_save({'model':student.state_dict(),'selection':student.selection,'accepted_val':accepted_val,
                     'next_round':ri+1,'planned_rounds':C.prune_rounds,'teacher_params':model_parameters(teacher),
                     'history':round_history},progress_file)
        logger.info('Round %d %s; val_acc %.3f%%',ri+1,'ACCEPTED' if accepted else 'REJECTED',accepted_val)
    except Exception as e:log_exception(f'pruning round {ri+1}',e);raise
atomic_save({'model':student.state_dict(),'selection':student.selection,'val_acc':accepted_val},RUN/'final_fp32.pt')
logger.info('Compression completed. Accepted validation accuracy %.3f%%',accepted_val)

## CLBM and PRISM comparison audit (optional)

Place `comparison.csv` and `comparison (1).csv` in the same working directory (or edit paths below). This table is for historical context only and **does not establish fair matched-FLOPs comparison** unless both runs use the same FLOPs convention and evaluation setup.

In [ ]:
from pathlib import Path
for csv_name in ('comparison.csv','comparison (1).csv'):
    p=Path(csv_name)
    if p.exists():
        try:
            print(f'\n{csv_name}');display(pd.read_csv(p))
        except Exception as e:
            log_exception(f'comparison CSV {csv_name}',e)
    else:
        print(f'Optional comparison file not found: {csv_name}')


## Final metrics, confusion matrix, plots and five random predictions

All test-set accuracy and classification metrics are calculated **once after model selection**. Model size is the exact serialized FP32 `state_dict` size. Latency is environment dependent. This notebook does not claim to measure energy without a physical power sampler.

In [ ]:
from sklearn.metrics import precision_recall_fscore_support,confusion_matrix,classification_report
try:
    if DEVICE.type=='cuda':torch.cuda.reset_peak_memory_stats()
    baseline_test=accuracy_and_loss(teacher,test_loader,collect=True)
    student_test=accuracy_and_loss(student,test_loader,collect=True)
    base_res=measure_resources(teacher,do_latency=True)
    student_res=measure_resources(student,do_latency=True)
    def class_metrics(result):
        p,r,f,_=precision_recall_fscore_support(result['y_true'],result['y_pred'],average='macro',zero_division=0)
        return {'precision_macro':float(p),'recall_macro':float(r),'f1_macro':float(f)}
    base_res.update({'test_accuracy':baseline_test['acc'],'test_loss':baseline_test['loss'],**class_metrics(baseline_test)})
    student_res.update({'test_accuracy':student_test['acc'],'test_loss':student_test['loss'],**class_metrics(student_test)})
    comparison=pd.DataFrame([{'model':'Original ResNet56',**base_res},{'model':'PRISM-compressed ResNet56',**student_res}])
    comparison.to_csv(RUN/'comparison.csv',index=False)
    summary={'parameter_compression_ratio':base_res['parameters']/student_res['parameters'],
             'flops_reduction_factor':base_res['flops']/student_res['flops'],
             'parameter_reduction_pct':100*(1-student_res['parameters']/base_res['parameters']),
             'flops_reduction_pct':100*(1-student_res['flops']/base_res['flops']),
             'accuracy_change_pp':student_test['acc']-baseline_test['acc'],
             'run_dir':str(RUN),'device':str(DEVICE),'smoke_test':C.smoke_test,
             'energy_measurement':'Not available (requires external power monitoring)'}
    (RUN/'summary.json').write_text(json.dumps(summary,indent=2),encoding='utf-8')
    print(comparison.to_string(index=False));print(json.dumps(summary,indent=2))
    cm=confusion_matrix(student_test['y_true'],student_test['y_pred'],labels=list(range(10)))
    np.save(RUN/'confusion_matrix.npy',cm)
    names=test_data.dataset.classes if isinstance(test_data,Subset) else test_data.classes
    (RUN/'classification_report.txt').write_text(classification_report(student_test['y_true'],student_test['y_pred'],target_names=names,zero_division=0),encoding='utf-8')
    fig,ax=plt.subplots(figsize=(8,6));im=ax.imshow(cm,cmap='Blues');plt.colorbar(im,ax=ax)
    ax.set(xticks=range(10),yticks=range(10),xticklabels=names,yticklabels=names,xlabel='Predicted class',ylabel='True class',title='PRISM confusion matrix')
    plt.setp(ax.get_xticklabels(),rotation=45,ha='right');fig.tight_layout();fig.savefig(RUN/'confusion_matrix.png',dpi=180);plt.show()
    fig,axes=plt.subplots(1,3,figsize=(14,4))
    for ax,key,ttl in zip(axes,('test_accuracy','parameters','flops'),('Top-1 accuracy (%)','Parameters','FLOPs (2 ops/MAC)')):
        values=comparison[key].values;ax.bar(['Original','PRISM'],values)
        ax.set_title(ttl);ax.bar_label(ax.containers[0],fmt='%.2f' if key=='test_accuracy' else '%.0f')
    fig.tight_layout();fig.savefig(RUN/'comparison.png',dpi=180);plt.show()
except Exception as e:log_exception('Final evaluation',e);raise

In [ ]:
# Reproducible five-image examples, original images de-normalized.
rng=np.random.default_rng(C.seed+8)
chosen=rng.choice(len(test_data),size=min(5,len(test_data)),replace=False)
student.eval();fig,axes=plt.subplots(1,len(chosen),figsize=(3*len(chosen),3.3))
axes=np.atleast_1d(axes)
for ax,k in zip(axes,chosen):
    x,y=test_data[int(k)]
    with torch.inference_mode():pred=int(student(x[None].to(DEVICE)).argmax(1).item())
    img=(x.permute(1,2,0)*torch.tensor(std)+torch.tensor(mean)).clamp(0,1).numpy()
    ax.imshow(img);ax.set_title(f'True: {names[y]}\nPred: {names[pred]}');ax.axis('off')
fig.suptitle('Five random held-out CIFAR-10 test images');fig.tight_layout();fig.savefig(RUN/'five_random_tests.png',dpi=180);plt.show()

## Paper-quality repeatability checklist

1. Repeat experiments for seeds 42, 123, 2026 with the same budget and document mean ± standard deviation.
2. Compare with a separately reproduced CLBM implementation under exactly the same preprocessing, validation split, FLOPs convention, and training budget. The published 44.41M figure may use a different FLOPs convention.
3. Evaluate at matched FLOPs and parameter budgets, not just maximum compression. Report accepted and rejected compression rounds and ablations.
4. For energy, connect an instrumented power monitor (e.g. NVML sampled during inference, if supported) and report the integration method; missing readings are **not** zero energy.
5. Inference latency is measured on the local machine and must not be compared with published timings from different devices.
6. This implementation uses an original *ResNet-56* with an option-A residual shortcut. Some public ResNet-56 code uses different shortcut conventions and may yield different baseline accuracy.
7. Current pipeline uses full precision FP32; there is **no quantization**, so any improvement is attributable to structured pruning plus recovery.
8. For deployment, explicitly save `final_fp32.pt` plus `config.json`; weights can be restored with `CIFARResNet56(checkpoint['selection'])`.

### Restore final compressed model
```python
checkpoint = torch.load(RUN/'final_fp32.pt', map_location=DEVICE, weights_only=False)
model = CIFARResNet56(checkpoint['selection']).to(DEVICE)
model.load_state_dict(checkpoint['model'])
model.eval()
```

**Novelty audit:** Classwise gradients, activation correlations, temporary masking and relational KD all have related prior work. A novel claim requires evidence that the *integrated, budgeted interaction- and recovery-aware policy* improves the Pareto frontier under identical teacher, splits, seeds and reporting conventions.

**Known limitations:** The two-channel interaction check is approximate and only applied to a bounded subset of modules. This notebook does not implement a full global knapsack solver or separate short recovery training for every candidate group. The final model can be less accurate than CLBM; report negative outcomes transparently.
